# import library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn #pip install scikit-learn
import scipy.stats as stats 
import csv
import requests
import statsmodels.api as sm
from statsmodels.formula.api import ols
import re

# option de visualisation output

In [ ]:
pd.set_option('display.max_columns', None) # affiche toutes les colonnes
pd.set_option('display.max_colwidth', None)  # Affiche les colonnes dans toute leur largeur
#pd.set_option('display.max_rows', None)# Affiche les lignes dans toute leur largeu

# Create df

In [ ]:
data2_short=pd.read_csv("data2_short.csv")

# Explore

In [ ]:
data2_short.head()

In [ ]:
data2_short.shape

In [ ]:
data2_short.info()

# analyse bivarie

In [ ]:
list_struct_NRJ = [
 'NumberofBuildings',
 'NumberofFloors',
 'PropertyGFATotal',
 'PropertyGFAParking',
 'PropertyGFABuilding(s)',
 'LargestPropertyUseTypeGFA',
 'SecondLargestPropertyUseTypeGFA',
 'ThirdLargestPropertyUseTypeGFA', 
 'ENERGYSTARScore',
'SiteEUIWN(kBtu/sf)', 'SiteEnergyUseWN(kBtu)', 'Electricity(kBtu)']

In [ ]:
corrMatrix22=data2_short[list_struct_NRJ].corr()# car considere zipcode comme variable quantitative alors que qualitative
corrMatrix22
sns.heatmap(corrMatrix22, annot=True)
plt.show()

# Modelisation predire conso Energie

## Definir problematique : 
BUT ANTHICIPER LES BESOINS DE CONSOMMATIONS DES BATIMENTS,
tenter de prédire la consommation totale d’énergie de bâtiments non destinés à l’habitation

donc, on regarde les données structurales des batiments, pour predire, selon la structure du batiment (surface etage...),et son utlisation quel sera sa consommation

## Preparation data avant 1ere Modélisation
### Choix des variables :
'SiteEnergyUseWN(kBtu)' climat dependant
'Electricity(kBtu)'
presente une belle correlation  

### Transfo Dataframe en numerique : class_bat

In [ ]:
# one shot encoding on this col
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore')#creating instance of one-hot-encoder
transform = encoder.fit_transform(data2_short[["Classe_bat"]])

In [ ]:
transform.toarray()

In [ ]:
#perform one-hot encoding on 'team' column 
encoder_df = pd.DataFrame(transform.toarray(),dtype ="Int32")

In [ ]:
encoder_df.shape

In [ ]:
encoder_df.columns = encoder. get_feature_names_out(["Classe_bat"])

In [ ]:
encoder_df.columns

In [ ]:
encoder_df

In [ ]:
encoder_df.nunique()

In [ ]:
# aNALYSE "SiteEUIWN(kBtu/sf)"
data2_short["SiteEnergyUseWN(kBtu)"].describe()

In [ ]:
# histogramme pour  col "SiteEUIWN(kBtu)"
fig=plt.figure(figsize=(20,10))
plt.title('SiteEnergyUseWN(kBtu)')
ax=plt.subplot(211)
sns.kdeplot(data2_short['SiteEnergyUseWN(kBtu)'])
plt.subplot(212, sharex=ax)
plt.boxplot(data2_short['SiteEnergyUseWN(kBtu)'],vert=False)
plt.show()

In [ ]:
#  Transfo en log.... surface and 
data2_short['LogSiteEnergyUseWN(kBtu)'] = np.log(data2_short['SiteEnergyUseWN(kBtu)'])

In [ ]:
list_new_feature2 = ['LogLargestPropertyUseTypeGFA','LogSecondLargestPropertyUseTypeGFA','LogThirdLargestPropertyUseTypeGFA']
# replace Nan by 0
data2_short[list_new_feature2]= data2_short[list_new_feature2].fillna(0)
data2_short[list_new_feature2]

In [ ]:
list_model_NRJ = [
 'CouncilDistrictCode',
 'Classe_Age',
 'NumberofBuildings',
 'NumberofFloors', 
 'ENERGYSTARScore',
'LogLargestPropertyUseTypeGFA', 
'LogSecondLargestPropertyUseTypeGFA',
'LogThirdLargestPropertyUseTypeGFA', 
'LogSiteEnergyUseWN(kBtu)']

In [ ]:
df_NRJ = data2_short[list_model_NRJ]

In [ ]:
df_NRJ.shape

In [ ]:
df_NRJ.isna().sum()

In [ ]:
df_NRJ.dropna(subset ="LogSiteEnergyUseWN(kBtu)",inplace = True)

In [ ]:
df_NRJ.isna().sum() # check


In [ ]:
df_NRJ.reset_index(inplace = True)
df_NRJ.drop(columns = ["index"],inplace =True)

In [ ]:
# Create a dataframe
df_model_NRJ = df_NRJ.join(encoder_df)

In [ ]:
df_model_NRJ

In [ ]:
X = df_model_NRJ.drop (columns = ["LogSiteEnergyUseWN(kBtu)"])
Y = df_model_NRJ["LogSiteEnergyUseWN(kBtu)"]

In [ ]:
X.shape,Y.shape

In [ ]:
X.isna().count()

#### Normalistation des datas

In [ ]:
from sklearn import preprocessing

In [ ]:
# normaliser tout entre 0 et 1 
norm = sklearn.preprocessing.normalize(X, norm='l2', axis=0, copy=True, return_norm=False) # axis = 0 means normalization amoung the col/features..better
x_norm = pd.DataFrame(norm, columns = X.columns)
x_norm 

### Creer un jeu de données test et train : Test train split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_norm,Y, test_size = 0.2, random_state = 1) #random state : ontrols how the data is shuffled before the split is implemented.

In [ ]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

## Regression lineaire 

###  Regression lineaire classique : baseline de regression des moindres carres

In [ ]:
from sklearn import linear_model
import numpy as np

# On crée un modèle de régression linéaire
lr = linear_model.LinearRegression() # droite f(x)

In [ ]:
# On entraîne ce modèle sur les données d'entrainement
lr.fit(x_train,y_train) # fit trouver les param qui minimise les moindres carres pour avoir le meilleur coef
print(lr.coef_) # 1 coef par feature X

In [ ]:
# Save coef in a dataframe
df_coef_lr =pd.DataFrame(lr.coef_, X.columns, columns = ['Coeff'])
df_coef_lr #High coef, means High impact for the prediction
#output : the highest coef are Number of Building and area, make sens for CO2 emission...

In [ ]:
# Resultats
print(lr.intercept_) # see the intercept (b or Beta0) of the regression equation

In [ ]:
y_pred_lr_test = lr.predict(x_test)
y_pred_lr_train = lr.predict(x_train)

In [ ]:
# On récupère l'erreur de norme 2 sur le jeu de données test comme baseline
baseline_error_test = np.mean((y_pred_lr_test - y_test) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur test, cad diff entre valeur obs et valeur predites 
baseline_error_train = np.mean((y_pred_lr_train - y_train) ** 2) # moy des erreurs quadratiques entre donnee et prediction sur training, cad diff entre valeur obs et valeur predites 

# Erreur quadratique = MSE: Mean Scare Error
print("Pour la régression linéaire basique : La MSE du training set est de : " + str(baseline_error_train.round(2)))
print("Pour la régression linéaire basique : La MSE du testing set est de : " + str(baseline_error_test.round(2)))

In [ ]:
# MSE via Sklearn
MSE_lr_test_sk = sklearn.metrics.mean_squared_error(y_test, y_pred_lr_test)
MSE_lr_train_sk = sklearn.metrics.mean_squared_error(y_train, y_pred_lr_train)
MSE_lr_test_sk, MSE_lr_train_sk # Meme resultats

In [ ]:
# RMSE via Sklearn : Racine de MSE, vire le carre
RMSE_lr_test_sk = sklearn.metrics.mean_squared_error(y_test, y_pred_lr_test, squared= False)
RMSE_lr_train_sk = sklearn.metrics.mean_squared_error(y_train, y_pred_lr_train, squared= False)
RMSE_lr_test_sk, RMSE_lr_train_sk 

In [ ]:
# Visualisation perf du model1
plt.scatter(y_train, y_pred_lr_train)
plt.xlabel("Consommation en Electricite :EUIWN(kBtu) Prédites sur training set")
plt.ylabel("Consommation en Electricite :EUIWN(kBtu) réelles sur training set")

# OUTPUT TOP !! 

In [ ]:
# Visualisation 
plt.scatter(y_test, y_pred_lr_test)
plt.xlabel("Consommation en Electricite/EUIWN(kBtu) Prédites sur jeu test")
plt.ylabel("Consommation en Electricite/EUIWN(kBtu) réelles sur jeu test")
# pas mieux sur testing set

### Dummy Regressor

In [ ]:
from sklearn import dummy
dum = dummy.DummyRegressor(strategy='mean')

In [ ]:

# Define Model
model1_dum = dummy.DummyRegressor().fit( x_train, y_train)
print("Dummy-R2 sur training set est de (0): "+str(model1_dum.score(x_train,y_train))+" : OK.")# output= 0

# Prédiction sur le jeu de test
y_pred_dum = model1_dum.predict(x_train)
y_pred_dum2 = model1_dum.predict(x_test)
print("Dummy-R2 sur testing set est de : "+str(model1_dum.score(x_test,y_test)))

In [ ]:
# Evaluate
print("MSE with Dummy Regressor sur le training set : {:.2f}".format(sklearn.metrics.mean_squared_error(y_train, y_pred_dum)) )
print("MSE with Dummy Regressor sur le testing set: {:.2f}".format(sklearn.metrics.mean_squared_error(y_test, y_pred_dum2)) )
print("RMSE with Dummy Regressor : {:.2f}".format(np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_pred_dum2)) ))

### Ameliorer la regression lineraire : Trouver le meilleur alpha/lambda as hyperparametres :
ridge
lasso
gridsearch sklearn

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

In [ ]:
# Coef lambda/alpha a tester
n_alphas = 200
alphas = np.logspace(-5, 5, n_alphas) # test different alpha, entre 10 puissance -5 et 10 puiss 5

#### Ridge

In [ ]:
from sklearn.linear_model import Ridge
# Regression sous contrainte lambda, test pour chaque lambda/alpha
ridge = linear_model.Ridge()
coefs = []
errors = []
for a in alphas:
    ridge.set_params(alpha=a)
    ridge.fit(x_train, y_train)
    coefs.append(ridge.coef_)
    errors.append([baseline_error_test, np.mean((ridge.predict(x_test) - y_test) ** 2)])

In [ ]:
# MSE
MSE_lr_ridge = np.mean((ridge.predict(x_test) - y_test) ** 2)
print("Pour la régression linéaire ridge : La MSE du testing set est de : " + str(MSE_lr_ridge.round(2)))

In [ ]:
np.argmin(errors,axis = 0)

In [ ]:
len(errors)

In [ ]:
# np.argmin(errors[:,1])
coef_opt = coefs[155]
coef_opt

In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs= np.array(errors)
plt.semilogx(alphas,erreurs[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev


In [ ]:
np.argmin(erreurs[:,1]) # on trouve l indice de l erreur mini qui ameliore l erreur

In [ ]:
# on veut retrouver alpha pour l erreur min
alpha_opt = alphas[np.argmin(erreurs[:,1])]
alpha_opt

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs)
plt.semilogx([alpha_opt, alpha_opt], [-4,4], 'r--')

In [ ]:
coef_opt = coefs[np.argmin(erreurs[:,1])]
coef_opt

#### Lasso : methode de selection de variables et de reduction de dimension supervise
remove celle norme 1 de B sup a t
region admissible +^- valeur absolue B

In [ ]:
from sklearn.linear_model import Lasso
# Regression sous contrainte lambda, test pour chaque lambda/alpha
lasso = linear_model.Lasso()

coefs2 = []
errors2 = []
for a in alphas:
    lasso.set_params(alpha=a)
    lasso.fit(x_train, y_train)
    coefs2.append(lasso.coef_)
    errors2.append([baseline_error_test, np.mean((lasso.predict(x_test) - y_test) ** 2)])

In [ ]:
# MSE
MSE_lr_lasso = np.mean((lasso.predict(x_test) - y_test) ** 2)
print("Pour la régression linéaire lasso : La MSE du testing set est de : " + str(MSE_lr_lasso.round(2)))


In [ ]:
# Visualisation de la difference entre (erreur quadratique : cest diff entre yobs et y predit (f(x)) au carree)
erreurs2= np.array(errors2)
plt.semilogx(alphas,erreurs2[:,1]) # valeur de erreur en fonction de alpha
plt.semilogx(alphas, erreurs2[:,0]) # baseline error, from linear regression (erreur quadratrique moy sur jeu test), ev
# output : il existe une valeur alpha/lambda qui ameliore la prediction (ameliore l erreur sur les données de test) 

In [ ]:
alpha_opt_lasso = alphas[np.argmin(erreurs2[:,1])]

# Chemin de regularisation
plt.semilogx(alphas, coefs2)
plt.semilogx([alpha_opt_lasso, alpha_opt_lasso], [-4,4], 'r--')

In [ ]:
# Chemin de regularisation
plt.semilogx(alphas, coefs2)
plt.xlim(.0001,.1)
plt.ylim(-.4,.2)
plt.semilogx([alpha_opt_lasso, alpha_opt_lasso], [-.5,.5], 'r--')

In [ ]:
alpha_opt_lasso 

In [ ]:
np.argmin(erreurs2[:,1])

In [ ]:
coef_opt2 = coefs2[np.argmin(erreurs2[:,1])]
coef_opt2

### Validation croisee Grid search

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# Define params
param_linear_grid = {'fit_intercept':[True,False], 'normalize':[True,False]}

In [ ]:
grid = GridSearchCV(lr,param_linear_grid,cv=10)
grid.fit(x_train, y_train)

In [ ]:

print("MSE gridsearch with kf=10 and lr method: %.2f"
              % np.mean((grid.predict(x_test) - y_test) ** 2))
print ("r2 / variance : ", grid.best_score_)

#### Lasso

In [ ]:
# Define params
param_lasso_grid = {'fit_intercept':[True,False], 'normalize':[True,False],'alpha': alphas}


In [ ]:
grid_lasso = GridSearchCV(lasso, param_lasso_grid,scoring='neg_mean_absolute_error',cv =10,verbose=3) 
grid_lasso.fit(x_train, y_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and lasso method: %.2f"
              % np.mean((grid_lasso.predict(x_test) - y_test) ** 2))
print ("r2 / variance : ", grid_lasso.best_score_)
print('Best Hyperparameters: %s' % grid_lasso.best_params_)

In [ ]:
df_res1_lasso_grid = pd.DataFrame(grid_lasso.cv_results_)
df_res1_lasso_grid["alpha"] = df_res1_lasso_grid.params.apply(lambda x: x["alpha"])


In [ ]:
df_res1_lasso_grid.columns

In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = plt.subplot()
df_res1_lasso_grid.set_index("alpha")[["mean_test_score"]].plot(ax=ax)
ax.set_xlabel("alpha")
ax.set_ylabel("mean_test_score")
plt.xlim([0,1])
#plt.xscale('log')

#### Ridge

In [ ]:
grid_ridge = GridSearchCV(ridge, param_lasso_grid,scoring='neg_mean_absolute_error',cv =10,verbose=3) 
grid_ridge.fit(x_train, y_train)

In [ ]:
# Results
print("MSE gridsearch with kf=10 and ridge method: %.2f"
              % np.mean((grid_ridge.predict(x_test) - y_test) ** 2))
print ("r2 / variance : ", grid_ridge.best_score_)
print('Best Hyperparameters: %s' % grid_ridge.best_params_)

In [ ]:
df_res1_ridge_grid = pd.DataFrame(grid_ridge.cv_results_)
df_res1_ridge_grid["alpha"] = df_res1_ridge_grid.params.apply(lambda x: x["alpha"])

In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = plt.subplot()
df_res1_ridge_grid.set_index("alpha")[["mean_test_score"]].plot(ax=ax)
ax.set_xlabel("alpha")
ax.set_ylabel("mean_test_score")
plt.xlim([0,2000])

In [ ]:
from sklearn.model_selection import KFold

In [ ]:
# Methode Kfold sklearn ( au lieu de faire un seuk train set, faire tour a tour)
 # K group = 10 pour nos 1500 datas
kf = KFold(n_splits=10) 

###### Cross Val score : MSE

In [ ]:
# How to estimate the accuray of a linear kernel (10 times)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer, r2_score
# return array of score (default = R2) of the estimator for each run
scores_lr = cross_val_score(lr, X, Y, cv=kf,scoring = make_scorer(mean_squared_error)) 
scores_lr # DONNE SCORE (MSE) A CHAQUE ITERATION


In [ ]:
print ("Le meilleur MSE, validation croisée LR : " + str ((min(scores_lr)).round(2)) )

In [ ]:
# Rigde
scores_ridge = cross_val_score(ridge, X, Y, cv=kf,scoring = make_scorer(mean_squared_error)) 
print ("Le meilleur MSE, validation croisée ridge: " + str ((min(scores_ridge)).round(2)) )

In [ ]:
# Lasso
scores_lasso = cross_val_score(lasso, X, Y, cv=kf,scoring = make_scorer(mean_squared_error) ) 
print ("Le meilleur MSE, validation croisée Lasso : " + str ((min(scores_lasso)).round(2)) )

In [ ]:
# report performance
print('MSE from lr, mean (sdt) : %.3f (%.3f)' % (np.mean(scores_lr), np.std(scores_lr)))

print('MSE from ridge, mean (sdt):  %.3f (%.3f)' % (np.mean(scores_ridge), np.std(scores_ridge)))

print('MSE from lasso, mean (sdt):  %.3f (%.3f)' % (np.mean(scores_lasso), np.std(scores_lasso)))

### Methode à Noyau = transfo non lineaire en lineaire

#### Linear SVR (Support Vector Machine for linear regression)

In [ ]:
# Import library
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

##### SVR libsvm implementation 

In [ ]:
#lr_SVR2 = make_pipeline(SVR(C=1.0, epsilon=0.2)) #StandardScaler()
lr_SVR = sklearn.svm.SVR(kernel='linear')
# Entraine le model
lr_SVR.fit(x_train, y_train)

In [ ]:
# Prediction sur testing set
y_pred_SVR = lr_SVR.predict(x_test)

In [ ]:
# MSE via Sklearn
MSE_SVR = sklearn.metrics.mean_squared_error(y_test, y_pred_SVR)
print ("le coef de correlation de la SVR sur testing set est de : MSE3_SVR = "+ str(MSE_SVR.round(2)))


In [ ]:
coef_SVR = lr_SVR.coef_
coef_SVR

##### SVR liblinear  implementation 

In [ ]:
# for large data set > 10k row, use linearSVR or SGDRegressor
LinearSVR = sklearn.svm.LinearSVR(epsilon=0.2)
# Entraine le model
LinearSVR.fit(x_train, y_train)

In [ ]:
# Prediction sur testing set
y_pred_LinearSVR = LinearSVR.predict(x_test)

In [ ]:
# MSE via Sklearn
MSE_LinearSVR = sklearn.metrics.mean_squared_error(y_test, y_pred_LinearSVR)
print ("MSE_LinearSVR = "+ str(MSE_LinearSVR.round(2)))


In [ ]:
coef_LinearSVR = LinearSVR.coef_
coef_LinearSVR

#### kRR Regression ridge kernel : RGF /Gaussien
identical to support SVR
KRR uses squared error loss while support vector regression uses -insensitive loss, both combined with l2 regularization

In [ ]:
from sklearn import kernel_ridge
kRR_GRF = kernel_ridge.KernelRidge(
    alpha=1.0, # valeur par défaut 
    kernel='rbf', # noyau Gaussien
    gamma=0.01)   # valeur de 1/(2 * sigma**2)
                                     
# entraîner le classifieur sur le jeu d'entrainement
kRR_GRF.fit(x_train, y_train)


In [ ]:
# prédire sur le jeu de test
y_pred_kRR = kRR_GRF.predict(x_test)

In [ ]:
# calculer la MSE sur le jeu de test
MSE_kRR_GRF = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR)
print ("MSE_kRR_RBF : {:.2f}".format(MSE_kRR_GRF))


In [ ]:
vector_kRR_GRF = kRR_GRF.dual_coef_
vector_kRR_GRF # Weight vector in kernel space

##### Gridsearch amelioration

In [ ]:
# valeurs du paramètre C, cf alpha de regression ridge (n=200, entre log de -5 et log 5)
alpha_range = np.logspace(-2, 2, 5)
# valeurs du paramètre gamma
gamma_range = np.logspace(-2, 1, 4)

# grille de paramètres
param_grid = {'alpha': alpha_range, 'gamma': gamma_range, 'kernel': ['linear', 'poly', 'rbf']}

# score pour sélectionner le modèle optimal
score = 'neg_mean_squared_error'

# initialiser la validation croisée
grid_pred_kRR = sklearn.model_selection.GridSearchCV(
        kernel_ridge.KernelRidge(kernel='rbf'),
        param_grid,
        cv=10,
        scoring=score)

# exécuter la validation croisée sur le jeu d'entraînement
grid_pred_kRR.fit(x_train, y_train)


# prédire sur le jeu de test avec le modèle sélectionné 
y_test_pred_kRR_grid = grid_pred_kRR.predict(x_test)

# calculer la RMSE correspondante
rmse_kRR_Grid = np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_test_pred_kRR_grid))

# calculer la MSE sur le jeu de test
MSE_kRR_GRF_Grid = sklearn.metrics.mean_squared_error(y_test, y_test_pred_kRR_grid)
print ("MSE_kRR_RBF_grid : {:.2f}".format(MSE_kRR_GRF_Grid))


In [ ]:
# entraîner le regresseur sur le jeu d'entrainement
kRR_GRF.fit(x_train, y_train)


In [ ]:
# prédire sur le jeu de test
y_pred3_kRR = kRR_GRF.predict(x_test)

In [ ]:
# calculer la MSE sur le jeu de test
MSE_kRR_GRF = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR)
print ("MSE_kRR_RBF : {:.2f}".format(MSE_kRR_GRF))


#### Polynomial-kRR

In [ ]:
kRR_poly1 = sklearn.kernel_ridge.KernelRidge(
    alpha=100, # valeur par défaut 
    kernel='poly', # noyau polynomial de degre 1
    gamma=0.1,
    degree=1)
                                     
# entraîner sur le jeu d'entrainement
kRR_poly1.fit(x_train, y_train)

# prédire sur le jeu de test
y_pred_kRR_poly1 = kRR_poly1.predict(x_test)


# calculer la MSE sur le jeu de test
MSE_kRR_poly1 = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR_poly1)
print ("MSE_kRR_poly1 : {:.4f}".format(MSE_kRR_poly1))


In [ ]:
vector_kRR_poly1 = kRR_poly1.dual_coef_
vector_kRR_poly1 # Weight vector in kernel space

In [ ]:
kRR_poly2 = sklearn.kernel_ridge.KernelRidge(
    alpha=100, # valeur par défaut 
    kernel='poly', # noyau polynomial de degre 2
    gamma=0.01,
    degree=2)
                                     
# entraîner sur le jeu d'entrainement
kRR_poly2.fit(x_train, y_train)

# prédire sur le jeu de test
y_pred_kRR_poly2 = kRR_poly2.predict(x_test)


# calculer la MSE sur le jeu de test
MSE_kRR_poly2 = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR_poly2)
print ("MSE_kRR_poly2 : {:.4f}".format(MSE_kRR_poly2))

In [ ]:
kRR_poly3 = sklearn.kernel_ridge.KernelRidge(
    alpha=100, # valeur par défaut 
    kernel='poly', # noyau polynomial de degre 3
    gamma=0.01,
    degree=3)
                                     
# entraîner sur le jeu d'entrainement
kRR_poly3.fit(x_train, y_train)

# prédire sur le jeu de test
y_pred_kRR_poly3 = kRR_poly3.predict(x_test)


# calculer la MSE sur le jeu de test
MSE_kRR_poly3 = sklearn.metrics.mean_squared_error(y_test, y_pred_kRR_poly3)
print ("MSE_kRR_poly3 : {:.4f}".format(MSE_kRR_poly3))

In [ ]:
deglist=[1,2,3,4,5,6,7,8,9,10,20]
MSE_krr_poly_deg=[]
for deg in deglist:
    kRR_poly5 = sklearn.kernel_ridge.KernelRidge(
        alpha=100, # valeur par défaut 
        kernel='poly', # noyau polynomial de degre 1
        gamma=0.01,
        degree=deg)                                 
    # entraîner sur le jeu d'entrainement
    kRR_poly5.fit(x_train, y_train)
    # prédire sur le jeu de test
    y_pred_kRR_poly5 = kRR_poly5.predict(x_test)
    # calculer la MSE sur le jeu de test
    MSE_krr_poly_deg.append(sklearn.metrics.mean_squared_error(y_test, y_pred_kRR_poly5))

plt.plot(deglist, MSE_krr_poly_deg, 'ob')
#plt.ylim([4.27, 4.28])
plt.xlabel('degree du polynome')
plt.ylabel('MSE kernel polynomial')

#### Methode ensembliste
##### GBRegressor 
GBR = GradientBoostingRegressor()

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
#sklearn.linear_model.SGDRegressor(loss='squared_error', *, penalty='l2', alpha=0.0001, l1_ratio=0.15, fit_intercept=True, max_iter=1000, tol=0.001, shuffle=True, verbose=0, epsilon=0.1, random_state=None, learning_rate='invscaling', eta0=0.01, power_t=0.25, early_stopping=False, validation_fraction=0.1, n_iter_no_change=5, warm_start=False, average=False)
GBR = sklearn.ensemble.GradientBoostingRegressor(random_state=1) #default criteirm friedman mse , mse upgraded
GBR.fit(x_train, y_train)

In [ ]:
GBR_pred = GBR.predict(x_test)

In [ ]:
GBR.score(x_test,y_test) #R2

In [ ]:
GBR.get_params()

In [ ]:
# MSE via Sklearn
MSE_GBR = sklearn.metrics.mean_squared_error(y_test, GBR_pred)
print ("MSE_GBR : {:.2f}".format(MSE_GBR))

In [ ]:
#FEATURE IMPORTANCE
GBR.feature_importances_

In [ ]:
# Gradient boosting
crossvalidation=KFold(n_splits=10,shuffle=True,random_state=1)
search_grid={'n_estimators':[100],'learning_rate':[.001,0.01,.1],'max_depth':[1,2,4],'subsample':[.5,.75,1],'random_state':[1]}
search=GridSearchCV(estimator=GBR,param_grid=search_grid,scoring='neg_mean_squared_error',n_jobs=1,cv=crossvalidation)

search.fit(x_train,y_train)



In [ ]:
search.best_score_, search.best_params_

In [ ]:
scores_GBR = cross_val_score(search, x_test,y_test, cv=kf,scoring = make_scorer(mean_squared_error)) 


In [ ]:
scores_GBR,np.mean(scores_GBR), np.min(scores_GBR)

In [ ]:
GBR_Pred_grid = search.predict(x_test)
MSE_GBR_grid = sklearn.metrics.mean_squared_error(y_test, GBR_Pred_grid)
print ("MSE_GBR : {:.2f}".format(MSE_GBR_grid))

##### RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Etablissement Model
RFR = RandomForestRegressor(max_depth=2, random_state=0)

# Entrainement
RFR.fit(x_train, y_train)

# Predicton
RFR_Pred = RFR.predict(x_test)

In [ ]:
RFR.get_params()

In [ ]:
RFR.apply(x_test) #return leaf indice

In [ ]:
RFR.feature_importances_

In [ ]:
# MSE via Sklearn
MSE_RFR = sklearn.metrics.mean_squared_error(y_test, RFR_Pred)
print ("MSE_RFR : {:.2f}".format(MSE_RFR))

##### Amelioration des modeles : gridsearch

In [ ]:
# RFR
param_grid = {
    'bootstrap': [True],
    'max_depth': [80, 90, 100, 110],
    'max_features': [2, 3],
    'min_samples_leaf': [3, 4, 5],
    'min_samples_split': [8, 10, 12],
    'n_estimators': [100]
}

rf = RandomForestRegressor()
grid_search_rf = GridSearchCV(estimator = rf, param_grid = param_grid, 
                          scoring='neg_mean_squared_error',n_jobs=1,cv=crossvalidation, verbose = 2)
grid_search_rf.fit(x_train, y_train)

In [ ]:
grid_search_rf.best_score_, grid_search_rf.best_params_

In [ ]:
scores_RFR = cross_val_score(rf, x_test,y_test, cv=crossvalidation,scoring = make_scorer(mean_squared_error)) 
scores_RFR,np.mean(scores_RFR), np.min(scores_RFR)

In [ ]:
RFR_Pred_grid = grid_search_rf.predict(x_test)
MSE_RFR_grid = sklearn.metrics.mean_squared_error(y_test, RFR_Pred_grid)
print ("MSE_RFR : {:.2f}".format(MSE_RFR_grid))

In [ ]:
RFR.score(x_test,y_test)

In [ ]:
parameters = {'C': [0.1, 1, 10, 100, 1000],  
              'gamma': [1, 0.1, 0.01, 0.001, 0.0001,'auto'], 
              'kernel': ['linear', 'poly', 'rbf', 'sigmoid']} 

grid_pred_multi = sklearn.model_selection.GridSearchCV(
        kernel_ridge.KernelRidge(kernel='rbf'),
        param_grid,
        cv=10,
        scoring=score)


Svm = GridSearchCV(grid_pred_multi, param_grid=parameters, cv=kf,verbose=10)

### Feature importance SHAP : SHapley Additive exPlanations
alternative to permutation feature importance
gbr svr only

In [ ]:
import shap
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# SVR LR
# vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()

explainer_LinearSVR = shap.Explainer(LinearSVR, x_train, feature_names=x_train.columns)
shap_values_LinearSVR = explainer_LinearSVR(x_test)


In [ ]:
shap.plots.beeswarm(shap_values_LinearSVR)

In [ ]:
shap.summary_plot(shap_values_LinearSVR,features=x_train, feature_names=x_train.columns,plot_type='bar')

In [ ]:
# GBR
explainer_GBR = shap.Explainer(GBR, x_train, feature_names=x_train.columns)
shap_values_GBR = explainer_GBR(x_test)
shap.plots.beeswarm(shap_values_GBR)

In [ ]:
shap.summary_plot(shap_values_GBR,features=x_test, feature_names=x_test.columns,plot_type='bar')